# 📊 DỰ ÁN 1 — XÂY DỰNG MÔ HÌNH PHÂN LOẠI VÀ DỰ BÁO RỦI RO KHÁCH HÀNG VAY VỐN

**Notebook 05/07 — Feature Engineering (Xây dựng đặc trưng)**

---

**🎯 Mục tiêu:** Xây dựng đặc trưng: tổng hợp thông tin từ các bảng phụ (bureau, previous_application, ...), tạo biến mới, mã hóa biến phân loại, chuẩn hóa/scale dữ liệu cho mô hình.

**📥 Input:** `data/raw/` (do notebook 03 chưa có nên đọc thẳng từ raw và làm sạch tối thiểu ở đây)

**📤 Output:** `data/processed/train_features.csv`, `data/processed/test_features.csv` — bảng đặc trưng hoàn chỉnh dùng cho huấn luyện và kiểm thử.
`models/scaler.pkl` — Đối tượng chuẩn hóa dữ liệu số để sử dụng lại cho demo app.

**🔗 Pipeline:** 04. EDA & Visualization → **05. Feature Engineering** → 06. Machine Learning

## 1. Chuẩn bị môi trường & Thiết lập cấu hình

Thiết lập các thư viện cần thiết, đường dẫn dữ liệu và cấu hình tối ưu hóa bộ nhớ để tránh lỗi tràn RAM khi làm việc với tập dữ liệu lớn.

In [1]:
import os
import gc
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Cấu hình pandas hiển thị
pd.set_option("display.max_columns", 150)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Cấu hình đường dẫn dữ liệu
DATA_RAW = next((p for p in [Path("data/raw"), Path("../data/raw")] if p.exists()), Path("../data/raw"))
DATA_PROCESSED = next((p for p in [Path("data/processed"), Path("../data/processed")] if p.exists()), Path("../data/processed"))
MODEL_DIR = next((p for p in [Path("models"), Path("../models")] if p.exists()), Path("../models"))

os.makedirs(DATA_PROCESSED, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Chế độ DEBUG để chạy thử nghiệm nhanh (True: lấy mẫu, False: chạy trên toàn bộ dữ liệu)
# Lưu ý: Mặc định đặt True để notebook có thể chạy hoàn chỉnh nhúng output nhanh trên máy RAM yếu (~2GB free).
# Đổi sang False để tạo bộ đặc trưng đầy đủ phục vụ huấn luyện mô hình cuối cùng.
DEBUG = True

print(f"Thư mục dữ liệu thô: {DATA_RAW.resolve()}")
print(f"Thư mục dữ liệu đầu ra: {DATA_PROCESSED.resolve()}")
print(f"Thư mục mô hình lưu scaler: {MODEL_DIR.resolve()}")
print(f"DEBUG mode: {DEBUG}")

Thư mục dữ liệu thô: C:\Users\bivibi\Downloads\credit-risk-classifier-main\data\raw
Thư mục dữ liệu đầu ra: C:\Users\bivibi\Downloads\credit-risk-classifier-main\data\processed
Thư mục mô hình lưu scaler: C:\Users\bivibi\Downloads\credit-risk-classifier-main\models
DEBUG mode: True


**Nhận xét:** Các đường dẫn dữ liệu và thư mục đầu ra đã được thiết lập đúng chuẩn. Chế độ `DEBUG` mặc định bật để xử lý mẫu dữ liệu nhanh chóng và an toàn về bộ nhớ RAM trên mọi cấu hình máy.

### 1.1. Hàm tiện ích tối ưu bộ nhớ và đọc dữ liệu phân đoạn

1. Hàm `reduce_mem_usage` giúp giảm kiểu dữ liệu số về mức nhỏ nhất có thể mà không làm mất thông tin.
2. Hàm `read_by_chunks` giúp đọc file CSV lớn theo từng chunk (phân đoạn) và chỉ giữ lại các dòng của khách hàng nằm trong danh sách lấy mẫu, giải phóng RAM tức thời cho các dòng không liên quan.

In [2]:
def reduce_mem_usage(df, verbose=True):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object and not pd.api.types.is_categorical_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        elif col_type == object:
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f"RAM giảm từ {start_mem:,.2f} MB xuống {end_mem:,.2f} MB ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)")
    return df

def read_by_chunks(filepath, target_ids=None, chunksize=100000):
    if target_ids is None:
        return pd.read_csv(filepath)
    
    chunks = []
    # Đọc theo chunk để lọc
    for chunk in pd.read_csv(filepath, chunksize=chunksize):
        filtered = chunk[chunk['SK_ID_CURR'].isin(target_ids)]
        chunks.append(filtered)
    return pd.concat(chunks, ignore_index=True)

## 2. Nạp dữ liệu chính & Làm sạch tối thiểu

Chúng ta sẽ nạp hai bảng đơn vay chính `application_train` và `application_test`, gộp chung lại thành `df_app` bằng cách thêm cột `is_train` để xử lý tiền xử lý cùng một lúc. Do notebook 03 (Data Cleaning) chưa được thực hiện độc lập, chúng ta sẽ thực hiện các bước làm sạch dữ liệu tối thiểu ngay tại đây.

In [3]:
# Nạp dữ liệu chính
if DEBUG:
    # Lấy mẫu 15,000 dòng train và 5,000 dòng test
    train = pd.read_csv(DATA_RAW / "application_train.csv", nrows=15000)
    test = pd.read_csv(DATA_RAW / "application_test.csv", nrows=5000)
else:
    train = pd.read_csv(DATA_RAW / "application_train.csv")
    test = pd.read_csv(DATA_RAW / "application_test.csv")

# Gán nhãn để phân biệt tập Train & Test sau khi gộp
train['is_train'] = 1
test['is_train'] = 0
test['TARGET'] = np.nan

# Gộp dữ liệu
df_app = pd.concat([train, test], ignore_index=True)
print(f"Kích thước bảng gộp đơn chính: {df_app.shape}")

# Danh sách mã khách hàng mục tiêu để lọc các bảng phụ
target_ids = set(df_app['SK_ID_CURR'])

# Giải phóng bộ nhớ của train/test cũ
del train, test
gc.collect()

# Làm sạch tối thiểu:
# 1. Cột DAYS_EMPLOYED có giá trị dị thường 365243 (1000 năm) -> Chuyển thành NaN và lưu cờ anom
df_app['DAYS_EMPLOYED_ANOM'] = (df_app['DAYS_EMPLOYED'] == 365243).astype(int)
df_app['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

# 2. Thay thế giới tính không xác định XNA bằng giới tính phổ biến nhất (F)
df_app['CODE_GENDER'].replace('XNA', 'F', inplace=True)

# 3. Chuyển đổi các cột Ngày (Days) âm sang dạng Dương (Positive) để dễ làm việc
days_cols = ['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']
for col in days_cols:
    if col in df_app.columns:
        df_app[col] = df_app[col].abs()

# Tối ưu hóa dung lượng
df_app = reduce_mem_usage(df_app)

Kích thước bảng gộp đơn chính: (20000, 123)


RAM giảm từ 18.92 MB xuống 3.90 MB (79.4% reduction)


**Nhận xét:** Bảng đơn vay chính đã được tải và gộp thành công với kích thước dòng và cột hợp lý. Giá trị dị thường trong cột `DAYS_EMPLOYED` đã được chuyển sang `NaN` và gắn cờ `DAYS_EMPLOYED_ANOM`, các cột biểu diễn khoảng thời gian dạng số âm cũng đã được đổi sang giá trị tuyệt đối dạng số dương để thống nhất.

## 3. Tổng hợp đặc trưng từ các bảng phụ (Aggregations)

Chúng ta sẽ đọc lần lượt từng bảng phụ, tính toán các chỉ số nghiệp vụ từ các bảng phụ và gom nhóm chúng theo khách hàng (`SK_ID_CURR`) để ghép vào bảng đơn chính. Mỗi bảng phụ sau khi được gom nhóm sẽ được downcast để tối ưu hóa bộ nhớ, sau đó giải phóng bảng thô.

### 3.1. Tổng hợp thông tin từ bảng Bureau và Bureau Balance

Bảng `bureau` ghi nhận các khoản vay cũ của khách hàng tại các tổ chức tín dụng khác (CIC), bảng `bureau_balance` theo dõi số dư và trạng thái đóng tiền hàng tháng cho từng khoản vay này.
Chúng ta sẽ:
- Nhóm `bureau_balance` theo `SK_ID_BUREAU` để tính tổng số tháng có số dư, số tháng bị quá hạn thanh toán (`STATUS` từ 1 đến 5), và nhóm nợ quá hạn cao nhất.
- Join kết quả này vào `bureau`.
- Tính hạn mức còn lại chưa sử dụng `amt_credit_sum_limit_unused` và tỷ lệ dư nợ trên hạn mức `debt_credit_ratio` trên `bureau`.
- Nhóm `bureau` theo `SK_ID_CURR` để tính toán các thống kê (mean, max, sum, count) cho từng khách hàng.

In [4]:
# 1. Đọc và lọc bureau
bureau = read_by_chunks(DATA_RAW / "bureau.csv", target_ids if DEBUG else None)
bureau_bureau_ids = set(bureau['SK_ID_BUREAU'])

# 2. Đọc và lọc bureau_balance theo SK_ID_BUREAU của các khoản vay trên
chunks = []
for chunk in pd.read_csv(DATA_RAW / "bureau_balance.csv", chunksize=100000):
    filtered = chunk[chunk['SK_ID_BUREAU'].isin(bureau_bureau_ids)]
    chunks.append(filtered)
bureau_balance = pd.concat(chunks, ignore_index=True)

# Đếm tổng số tháng và số tháng quá hạn
bureau_balance['is_overdue'] = bureau_balance['STATUS'].isin(['1', '2', '3', '4', '5']).astype(int)
# Ánh xạ trạng thái quá hạn sang số để tìm mức lớn nhất
status_map = {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, 'C': 0, 'X': 0}
bureau_balance['status_num'] = bureau_balance['STATUS'].map(status_map).fillna(0).astype(int)

# Gom nhóm theo sk_id_bureau
bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg(
    bb_total_months=('MONTHS_BALANCE', 'count'),
    bb_overdue_months=('is_overdue', 'sum'),
    bb_max_overdue_status=('status_num', 'max')
).reset_index()

bb_agg = reduce_mem_usage(bb_agg, verbose=False)
del bureau_balance
gc.collect()

# Join kết quả của bureau_balance vào bureau
bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')

# Xử lý các cột đặc trưng của bureau
bureau['amt_credit_sum_limit_unused'] = bureau['AMT_CREDIT_SUM'].fillna(0) - bureau['AMT_CREDIT_SUM_DEBT'].fillna(0)
bureau['debt_credit_ratio'] = np.where(bureau['AMT_CREDIT_SUM'] > 0, bureau['AMT_CREDIT_SUM_DEBT'] / bureau['AMT_CREDIT_SUM'], 0)

# Gom nhóm bureau theo SK_ID_CURR
bureau_num_agg = bureau.groupby('SK_ID_CURR').agg(
    bureau_loan_count=('SK_ID_BUREAU', 'count'),
    bureau_active_loans=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    bureau_closed_loans=('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
    bureau_total_debt=('AMT_CREDIT_SUM_DEBT', 'sum'),
    bureau_total_limit=('AMT_CREDIT_SUM', 'sum'),
    bureau_avg_debt_credit_ratio=('debt_credit_ratio', 'mean'),
    bureau_max_overdue_days=('CREDIT_DAY_OVERDUE', 'max'),
    bureau_avg_overdue_months=('bb_overdue_months', 'mean'),
    bureau_max_overdue_status=('bb_max_overdue_status', 'max')
).reset_index()

bureau_num_agg = reduce_mem_usage(bureau_num_agg)

# Ghép vào df_app
df_app = df_app.merge(bureau_num_agg, on='SK_ID_CURR', how='left')

del bureau, bureau_num_agg, bb_agg
gc.collect()
print(f"Kích thước sau khi join Bureau: {df_app.shape}")

RAM giảm từ 1.31 MB xuống 0.38 MB (71.2% reduction)
Kích thước sau khi join Bureau: (20000, 133)


**Nhận xét:** Đã tổng hợp thành công đặc trưng từ `bureau` và `bureau_balance` theo từng `SK_ID_CURR` và ghép vào bảng đơn chính. Các đặc trưng mới như số khoản vay ngoài (`bureau_loan_count`), số khoản vay đang hoạt động (`bureau_active_loans`), tổng dư nợ nợ ngoài (`bureau_total_debt`), tỷ lệ nợ nợ/hạn mức trung bình (`bureau_avg_debt_credit_ratio`) và số tháng quá hạn trung bình (`bureau_avg_overdue_months`) sẽ giúp phản ánh lịch sử uy tín của khách hàng từ hệ thống CIC.

### 3.2. Tổng hợp thông tin từ bảng Previous Application

Bảng `previous_application` lưu trữ lịch sử các đơn đăng ký vay trước đây của khách hàng tại Home Credit. Chúng ta sẽ tính tỷ lệ giữa số tiền được duyệt so với số tiền đăng ký (`approved_applied_ratio`), tỷ lệ hồ sơ được duyệt (`is_approved`), bị từ chối (`is_refused`), bị hủy (`is_canceled`), sau đó gom nhóm theo khách hàng.

In [5]:
prev_app = read_by_chunks(DATA_RAW / "previous_application.csv", target_ids if DEBUG else None)

# Tính các biến phụ trợ ở cấp đơn vay cũ
prev_app['approved_applied_ratio'] = np.where(prev_app['AMT_APPLICATION'] > 0, prev_app['AMT_CREDIT'] / prev_app['AMT_APPLICATION'], 0)
prev_app['is_approved'] = (prev_app['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
prev_app['is_refused'] = (prev_app['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)
prev_app['is_canceled'] = (prev_app['NAME_CONTRACT_STATUS'] == 'Canceled').astype(int)

# Gom nhóm theo SK_ID_CURR
prev_agg = prev_app.groupby('SK_ID_CURR').agg(
    prev_app_count=('SK_ID_PREV', 'count'),
    prev_approved_count=('is_approved', 'sum'),
    prev_refused_count=('is_refused', 'sum'),
    prev_canceled_count=('is_canceled', 'sum'),
    prev_avg_approved_applied_ratio=('approved_applied_ratio', 'mean'),
    prev_total_amt_credit=('AMT_CREDIT', 'sum'),
    prev_avg_amt_credit=('AMT_CREDIT', 'mean'),
    prev_avg_annuity=('AMT_ANNUITY', 'mean')
).reset_index()

# Tính tỷ lệ duyệt (Approval rate)
prev_agg['prev_approval_rate'] = np.where(prev_agg['prev_app_count'] > 0, prev_agg['prev_approved_count'] / prev_agg['prev_app_count'], 0)

prev_agg = reduce_mem_usage(prev_agg)

# Ghép vào df_app
df_app = df_app.merge(prev_agg, on='SK_ID_CURR', how='left')

del prev_app, prev_agg
gc.collect()
print(f"Kích thước sau khi join Previous Application: {df_app.shape}")

RAM giảm từ 1.46 MB xuống 0.44 MB (70.0% reduction)
Kích thước sau khi join Previous Application: (20000, 142)


**Nhận xét:** Thông tin về các khoản vay trước đó của khách hàng tại Home Credit đã được tích hợp. Các chỉ số như tổng số đơn vay cũ (`prev_app_count`), tỷ lệ hồ sơ được duyệt (`prev_approval_rate`), số đơn bị từ chối (`prev_refused_count`) cung cấp thông tin quý giá về mối quan hệ tín dụng trong quá khứ của khách hàng đối với Home Credit.

### 3.3. Tổng hợp thông tin từ bảng Installments Payments

Bảng `installments_payments` ghi lại lịch sử thanh toán thực tế của các khoản vay trả góp tại Home Credit. Chúng ta sẽ tính số ngày đóng trễ (`days_late`) và số tiền trả thiếu (`amt_underpaid`), sau đó gom nhóm theo khách hàng để đo lường thói quen trả nợ định kỳ.

In [6]:
installments = read_by_chunks(DATA_RAW / "installments_payments.csv", target_ids if DEBUG else None)

# Tính số ngày trả muộn và số tiền đóng thiếu cho từng lần thanh toán
installments['days_late'] = (installments['DAYS_ENTRY_PAYMENT'] - installments['DAYS_INSTALMENT']).clip(lower=0)
installments['amt_underpaid'] = (installments['AMT_INSTALMENT'] - installments['AMT_PAYMENT']).clip(lower=0)

# Gom nhóm theo SK_ID_CURR
inst_agg = installments.groupby('SK_ID_CURR').agg(
    inst_total_records=('SK_ID_PREV', 'count'),
    inst_late_count=('days_late', lambda x: (x > 0).sum()),
    inst_underpaid_count=('amt_underpaid', lambda x: (x > 0).sum()),
    inst_avg_days_late=('days_late', 'mean'),
    inst_max_days_late=('days_late', 'max'),
    inst_total_amt_required=('AMT_INSTALMENT', 'sum'),
    inst_total_amt_paid=('AMT_PAYMENT', 'sum'),
    inst_avg_underpaid=('amt_underpaid', 'mean')
).reset_index()

inst_agg = reduce_mem_usage(inst_agg)

# Ghép vào df_app
df_app = df_app.merge(inst_agg, on='SK_ID_CURR', how='left')

del installments, inst_agg
gc.collect()
print(f"Kích thước sau khi join Installments Payments: {df_app.shape}")

RAM giảm từ 1.32 MB xuống 0.46 MB (65.3% reduction)
Kích thước sau khi join Installments Payments: (20000, 150)


**Nhận xét:** Các đặc trưng trả nợ định kỳ đã được gom nhóm và tích hợp thành công. Tỷ lệ số lần đóng tiền trễ (`inst_late_count`) và đóng thiếu tiền (`inst_underpaid_count`), số ngày trễ tối đa (`inst_max_days_late`) là những tín hiệu cảnh báo cực kỳ nhạy bén cho các mô hình dự báo nợ xấu.

### 3.4. Tổng hợp thông tin từ bảng POS CASH Balance

Bảng `POS_CASH_balance` theo dõi số dư và trạng thái các khoản vay trả góp tại POS (điện thoại, điện máy...) và khoản vay tiền mặt tiêu dùng. Chúng ta sẽ tính số lần và số ngày trễ hạn đóng POS/CASH hàng tháng.

In [7]:
pos_cash = read_by_chunks(DATA_RAW / "POS_CASH_balance.csv", target_ids if DEBUG else None)

# Gom nhóm theo SK_ID_CURR
pos_agg = pos_cash.groupby('SK_ID_CURR').agg(
    pos_total_records=('SK_ID_PREV', 'count'),
    pos_avg_installments=('CNT_INSTALMENT', 'mean'),
    pos_avg_future_installments=('CNT_INSTALMENT_FUTURE', 'mean'),
    pos_max_dpd=('SK_DPD', 'max'),
    pos_overdue_months=('SK_DPD', lambda x: (x > 0).sum())
).reset_index()

pos_agg = reduce_mem_usage(pos_agg)

# Ghép vào df_app
df_app = df_app.merge(pos_agg, on='SK_ID_CURR', how='left')

del pos_cash, pos_agg
gc.collect()
print(f"Kích thước sau khi join POS CASH: {df_app.shape}")

RAM giảm từ 0.87 MB xuống 0.24 MB (72.9% reduction)
Kích thước sau khi join POS CASH: (20000, 155)


**Nhận xét:** Đã tổng hợp xong đặc trưng từ bảng `POS_CASH_balance`. Biến `pos_max_dpd` (số ngày quá hạn tối đa) và `pos_overdue_months` (số tháng bị quá hạn thanh toán) cho thấy tần suất chậm thanh toán của khách hàng đối với các món trả góp POS.

### 3.5. Tổng hợp thông tin từ bảng Credit Card Balance

Bảng `credit_card_balance` lưu trữ số dư và lịch sử giao dịch thẻ tín dụng hàng tháng của khách hàng tại Home Credit. Chúng ta sẽ tính tỷ lệ sử dụng hạn mức thẻ tín dụng (`card_utilization_ratio`) và các thống kê liên quan đến hạn mức/dư nợ/số ngày chậm trả nợ thẻ.

In [8]:
credit_card = read_by_chunks(DATA_RAW / "credit_card_balance.csv", target_ids if DEBUG else None)

# Tính tỷ lệ sử dụng hạn mức thẻ tín dụng
credit_card['card_utilization_ratio'] = np.where(
    credit_card['AMT_CREDIT_LIMIT_ACTUAL'] > 0,
    credit_card['AMT_BALANCE'] / credit_card['AMT_CREDIT_LIMIT_ACTUAL'],
    0
)

# Gom nhóm theo SK_ID_CURR
cc_agg = credit_card.groupby('SK_ID_CURR').agg(
    cc_total_records=('SK_ID_PREV', 'count'),
    cc_avg_balance=('AMT_BALANCE', 'mean'),
    cc_max_balance=('AMT_BALANCE', 'max'),
    cc_avg_limit=('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
    cc_avg_drawings=('AMT_DRAWINGS_CURRENT', 'mean'),
    cc_max_dpd=('SK_DPD', 'max'),
    cc_overdue_months=('SK_DPD', lambda x: (x > 0).sum()),
    cc_avg_utilization=('card_utilization_ratio', 'mean')
).reset_index()

cc_agg = reduce_mem_usage(cc_agg)

# Ghép vào df_app
df_app = df_app.merge(cc_agg, on='SK_ID_CURR', how='left')

del credit_card, cc_agg
gc.collect()
print(f"Kích thước sau khi join Credit Card Balance: {df_app.shape}")

RAM giảm từ 0.41 MB xuống 0.15 MB (62.5% reduction)
Kích thước sau khi join Credit Card Balance: (20000, 163)


**Nhận xét:** Đặc trưng từ bảng `credit_card_balance` đã được gom nhóm và join vào dữ liệu chính. Tỷ lệ sử dụng hạn mức thẻ trung bình (`cc_avg_utilization`) và số dư nợ thẻ trung bình (`cc_avg_balance`) giúp phản ánh áp lực nợ thẻ tín dụng của khách hàng.

## 4. Tạo đặc trưng tương tác (Domain Features) từ bảng chính

Các biến tương tác đóng vai trò cực kỳ quan trọng đối với rủi ro tín dụng. Chúng ta sẽ tạo ra một số đặc trưng nghiệp vụ như tỷ lệ chi trả định kỳ trên thu nhập (DTI), thu nhập bình quân đầu người, và tích hợp các điểm số đánh giá từ nguồn tín dụng độc lập (`EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`).

In [9]:
# Tỷ lệ thu nhập trên khoản vay
df_app['INCOME_CREDIT_PERC'] = df_app['AMT_INCOME_TOTAL'] / df_app['AMT_CREDIT']

# Tỷ lệ số tiền phải trả định kỳ trên thu nhập (DTI - Debt to Income)
df_app['ANNUITY_INCOME_PERC'] = df_app['AMT_ANNUITY'] / df_app['AMT_INCOME_TOTAL']

# Tỷ lệ thanh toán định kỳ trên tổng khoản vay
df_app['PAYMENT_RATE'] = df_app['AMT_ANNUITY'] / df_app['AMT_CREDIT']

# Thu nhập bình quân đầu người của gia đình khách hàng
df_app['INCOME_PER_PERSON'] = df_app['AMT_INCOME_TOTAL'] / df_app['CNT_FAM_MEMBERS']

# Tỷ lệ con cái trên tổng số thành viên gia đình
df_app['CHILDREN_RATIO'] = np.where(df_app['CNT_FAM_MEMBERS'] > 0, df_app['CNT_CHILDREN'] / df_app['CNT_FAM_MEMBERS'], 0)

# Các đặc trưng kết hợp từ điểm tín dụng nguồn ngoài (EXT_SOURCE_1, 2, 3)
ext_sources = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
df_app['EXT_SOURCES_MEAN'] = df_app[ext_sources].mean(axis=1)
df_app['EXT_SOURCES_STD'] = df_app[ext_sources].std(axis=1).fillna(0)
df_app['EXT_SOURCES_MUL'] = df_app['EXT_SOURCE_1'].fillna(1) * df_app['EXT_SOURCE_2'].fillna(1) * df_app['EXT_SOURCE_3'].fillna(1)

# Tối ưu bộ nhớ sau khi tạo đặc trưng mới
df_app = reduce_mem_usage(df_app)
print(f"Kích thước bảng đặc trưng sau khi tạo biến tương tác: {df_app.shape}")

RAM giảm từ 8.25 MB xuống 6.19 MB (25.0% reduction)
Kích thước bảng đặc trưng sau khi tạo biến tương tác: (20000, 171)


**Nhận xét:** Các biến tương tác nghiệp vụ tài chính đã được bổ sung thành công vào bảng tổng hợp. Việc kết hợp ba nguồn điểm số tín dụng ngoài (`EXT_SOURCE_1, 2, 3`) thành các thống kê trung bình, độ lệch chuẩn và tích nhân sẽ cung cấp một tín hiệu mạnh mẽ đại diện cho xếp hạng tín dụng tổng hợp của khách hàng.

## 5. Tiền xử lý dữ liệu (Preprocessing)

Trong phần này, chúng ta sẽ thực hiện mã hóa các biến phân loại và chuẩn hóa các biến số. Việc chuẩn hóa sẽ giúp cải thiện sự hội tụ của nhiều thuật toán và chuẩn bị sẵn dữ liệu sạch để nạp trực tiếp vào các mô hình trong notebook tiếp theo.

### 5.1. Mã hóa các biến phân loại (Categorical Encoding)

Chúng ta sẽ thực hiện One-Hot Encoding cho các biến phân loại có kiểu dữ liệu là `'category'` hoặc `'object'`.

In [10]:
# Xác định các cột phân loại
categorical_cols = df_app.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Số lượng cột phân loại cần mã hóa: {len(categorical_cols)}")

# Thực hiện One-Hot Encoding
df_app = pd.get_dummies(df_app, columns=categorical_cols, dummy_na=False, dtype=int)
print(f"Kích thước bảng đặc trưng sau khi mã hóa One-Hot: {df_app.shape}")

Số lượng cột phân loại cần mã hóa: 16
Kích thước bảng đặc trưng sau khi mã hóa One-Hot: (20000, 292)


**Nhận xét:** Đã hoàn tất mã hóa One-Hot cho toàn bộ biến phân loại trong bảng dữ liệu. Số lượng đặc trưng tổng cộng hiện tại của bộ dữ liệu sau khi mở rộng các biến One-Hot là hợp lý.

### 5.2. Tách dữ liệu Train / Test

Để chuẩn bị chuẩn hóa và lưu trữ, chúng ta cần tách bảng gộp `df_app` ngược lại thành tập Train và tập Test. Tập Train sẽ chứa nhãn `TARGET` và là cơ sở để huấn luyện mô hình. Tập Test không chứa nhãn `TARGET` (bỏ cột `TARGET` vì giá trị của nó là NaN).

In [11]:
# Tách tập Train và tập Test
train_df = df_app[df_app['is_train'] == 1].drop(columns=['is_train'])
test_df = df_app[df_app['is_train'] == 0].drop(columns=['is_train', 'TARGET'])

print(f"Kích thước tập Train: {train_df.shape}")
print(f"Kích thước tập Test: {test_df.shape}")

# Giải phóng bộ nhớ của bảng gộp lớn
del df_app
gc.collect()

Kích thước tập Train: (15000, 291)
Kích thước tập Test: (5000, 290)


12

**Nhận xét:** Bộ dữ liệu đã được phân tách thành tập Train và tập Test riêng biệt. Cột phân biệt `is_train` đã được loại bỏ để trả lại cấu trúc chuẩn.

### 5.3. Chuẩn hóa dữ liệu dạng số (Scaling)

Chúng ta sử dụng `StandardScaler` từ `scikit-learn` để chuẩn hóa các cột số (trừ cột định danh `SK_ID_CURR` và biến mục tiêu `TARGET`). Để tránh rò rỉ thông tin dữ liệu (data leakage), ta chỉ khớp (`fit`) scaler trên tập Train, sau đó áp dụng phép biến đổi (`transform`) lên cả hai tập Train và Test. Sau khi chuẩn hóa xong, chúng ta sẽ lưu đối tượng `StandardScaler` vào thư mục `models/` để sử dụng lại cho demo dự đoán.

In [12]:
# Xác định các cột số cần chuẩn hóa (loại bỏ SK_ID_CURR và TARGET)
exclude_cols = ['SK_ID_CURR', 'TARGET']
numeric_cols = [col for col in train_df.columns if col not in exclude_cols and not col.startswith('is_train') and train_df[col].dtype in ['int8', 'int16', 'int32', 'int64', 'float16', 'float32', 'float64', 'int']]

print(f"Số lượng đặc trưng dạng số cần chuẩn hóa: {len(numeric_cols)}")

# Khởi tạo và fit StandardScaler trên tập Train
scaler = StandardScaler()
scaler.fit(train_df[numeric_cols].fillna(train_df[numeric_cols].median())) # Điền khuyết tạm thời bằng median khi fit scale

# Chuẩn hóa tập Train
train_scaled = train_df.copy()
train_scaled[numeric_cols] = scaler.transform(train_scaled[numeric_cols].fillna(train_scaled[numeric_cols].median()))

# Chuẩn hóa tập Test
test_scaled = test_df.copy()
test_scaled[numeric_cols] = scaler.transform(test_scaled[numeric_cols].fillna(train_df[numeric_cols].median())) # Điền khuyết bằng median của Train

# Lưu đối tượng Scaler để phục vụ Demo App
scaler_path = MODEL_DIR / "scaler.pkl"
joblib.dump(scaler, scaler_path)
print(f"Đã lưu scaler vào: {scaler_path.resolve()}")

Số lượng đặc trưng dạng số cần chuẩn hóa: 289


Đã lưu scaler vào: C:\Users\bivibi\Downloads\credit-risk-classifier-main\models\scaler.pkl


**Nhận xét:** Đã chuẩn hóa thành công các đặc trưng dạng số và lưu trữ đối tượng `StandardScaler` vào `models/scaler.pkl`. Việc chuẩn hóa giúp đưa tất cả các đặc trưng về cùng một phân phối (trung bình bằng 0 và độ lệch chuẩn bằng 1), điều này rất quan trọng để đảm bảo tính ổn định và chính xác khi so sánh các mô hình tuyến tính hoặc các mô hình dựa trên khoảng cách.

## 6. Xuất dữ liệu đã xử lý ra thư mục data/processed/

Cuối cùng, chúng ta sẽ lưu các tập dữ liệu đặc trưng đã hoàn thành tiền xử lý ra file CSV trong thư mục `data/processed/`. Các file này sẽ được nạp trực tiếp trong notebook `06_machine_learning.ipynb` để huấn luyện các mô hình Machine Learning.

In [13]:
# Thiết lập đường dẫn lưu trữ
train_output_path = DATA_PROCESSED / "train_features.csv"
test_output_path = DATA_PROCESSED / "test_features.csv"

# Lưu các file đặc trưng
print("Đang ghi tập Train ra file CSV...")
train_scaled.to_csv(train_output_path, index=False)
print(f"Đã ghi xong tập Train vào: {train_output_path.resolve()} (Dung lượng: {train_output_path.stat().st_size / 1024**2:,.2f} MB)")

print("Đang ghi tập Test ra file CSV...")
test_scaled.to_csv(test_output_path, index=False)
print(f"Đã ghi xong tập Test vào: {test_output_path.resolve()} (Dung lượng: {test_output_path.stat().st_size / 1024**2:,.2f} MB)")

# Giải phóng bộ nhớ
del train_scaled, test_scaled, train_df, test_df
gc.collect()

Đang ghi tập Train ra file CSV...


Đã ghi xong tập Train vào: C:\Users\bivibi\Downloads\credit-risk-classifier-main\data\processed\train_features.csv (Dung lượng: 83.67 MB)
Đang ghi tập Test ra file CSV...


Đã ghi xong tập Test vào: C:\Users\bivibi\Downloads\credit-risk-classifier-main\data\processed\test_features.csv (Dung lượng: 27.86 MB)


0

**Nhận xét:** Các tệp đặc trưng `train_features.csv` và `test_features.csv` đã được lưu thành công vào thư mục `data/processed/`. Pipeline feature engineering hoạt động hoàn hảo và sẵn sàng nạp dữ liệu sạch cho các bước huấn luyện tiếp theo.

## 7. Tổng kết

Trong notebook này, chúng ta đã hoàn tất quy trình Feature Engineering (Xây dựng đặc trưng) chi tiết cho bộ dữ liệu Home Credit:
1. **Làm sạch tối thiểu**: Khắc phục các điểm bất thường như giá trị `365243` ngày làm việc và các giá trị không xác định khác.
2. **Tổng hợp dữ liệu phụ**: Tổng hợp và xây dựng các biến đặc trưng giá trị từ 5 bảng dữ liệu phụ (bureau & bureau_balance, previous_application, installments_payments, POS_CASH_balance, credit_card_balance) về cấp độ khách hàng (`SK_ID_CURR`).
3. **Xây dựng biến nghiệp vụ (Domain Features)**: Thiết lập các đặc trưng tài chính thiết yếu (tỷ lệ thanh toán nợ định kỳ, tỷ lệ thu nhập/khoản vay, điểm tín dụng ngoài kết hợp...).
4. **Tiền xử lý & Chuẩn hóa**: Mã hóa One-Hot các biến phân loại và chuẩn hóa các biến số bằng `StandardScaler`, tránh rò rỉ dữ liệu (data leakage) giữa tập Train/Test và lưu đối tượng chuẩn hóa thành `models/scaler.pkl`.
5. **Xuất kết quả**: Tạo thành công `train_features.csv` và `test_features.csv` lưu trong `data/processed/`.

**Bước tiếp theo:** Chuyển sang notebook [06_machine_learning.ipynb](file:///c:/Users/bivibi/Downloads/credit-risk-classifier-main/notebooks/06_machine_learning.ipynb) để bắt đầu xây dựng, huấn luyện và so sánh các mô hình Machine Learning phân loại rủi ro tín dụng.